# Benchmark hidrelétrico sintético v1.0.0

Este notebook reproduz primeiro o balanço hídrico determinístico, sem executar o GIVP. Os parâmetros são sintéticos e perturbados, ancorados em faixas públicas associadas a Monte Claro e 14 de Julho. Nenhum código, série histórica, tabela operacional ou coeficiente proprietário do SOG2 foi incorporado. A avaliação de independência técnica não substitui análise jurídica de patente, contratos ou NDA.

## Variáveis e protocolo

O índice $u$ identifica a usina A ou B e $t$ identifica uma das 24 horas. $y_{u,t}$ é a afluência incremental; $I_{u,t}$, a afluência total; $Q_{u,t}$, a vazão turbinada; $S_{u,t}$, o vertimento; $D_{u,t}$, a defluência; $V_{u,t}$, o volume; $n^{up}_{u,t}$ e $n^{down}_{u,t}$, os níveis de montante e jusante; $H_{u,t}$, a queda; e $P_{u,t}$, a potência.

Cada cenário cruza seis níveis constantes de potência em A com seis em B: desligada, mínima, 25%, 50%, 75% e máxima. São 36 casos por cenário e 252 casos no total.

## Equações simplificadas

Afluência em A:

$$I_{A,t}=y_{A,t}$$

Afluência em B, com uma hora de atraso:

$$I_{B,t}=y_{B,t}+D_{A,t-1}$$

Defluência total:

$$D_{u,t}=Q_{u,t}+S_{u,t}$$

Conservação de massa, com $\Delta V=3600/10^6$ para períodos horários:

$$V_{u,t+1}=V_{u,t}+\Delta V(I_{u,t}-Q_{u,t}-S_{u,t})$$

Nível de montante, função quartica sintética do volume normalizado $x$:

$$n^{up}=n^{min}+(n^{maximorum}-n^{min})\sum_{k=0}^{4}a_kx^k$$

Nível de jusante, função quartica sintética da defluência normalizada $z$:

$$n^{down}=a^{down}+r^{down}\sum_{k=0}^{4}b_kz^k$$

Queda líquida:

$$H=\max(0,n^{up}-n^{down})$$

Potência, com $g=9{,}81\,m/s^2$:

$$P=0{,}00981\,\eta H Q$$

A inversão potência-vazão usa bisseção determinística. O mesmo passo hidráulico calcula volume, níveis, vertimento e propagação nos simuladores por vazão e por potência. Metas inviáveis permanecem no benchmark com potência realizada, déficit e status limitante.

In [ ]:
import pandas as pd
from givp.examples.synthetic_hydropower.benchmark import (
    load_deterministic_definition,
    load_frozen_inflows,
    run_deterministic_benchmark,
)
from givp.examples.synthetic_hydropower.model import config as model_config
from givp.examples.synthetic_hydropower.paths import project_root
from IPython.display import Image, display

assert abs(model_config.WATER_POWER_FACTOR_MW - 0.00981) < 1e-12

## Execução determinística

Os caminhos são resolvidos pelo pacote instalado. `base.json` contém somente as duas usinas; horizonte, cenários, seeds e tolerâncias vêm da definição do protocolo `deterministic_balance`. As afluências são compartilhadas pelos dois protocolos e lidas do CSV congelado, não regeneradas.

In [ ]:
benchmark_dir = project_root() / "benchmarks" / "v1.0.0"
deterministic_dir = benchmark_dir / "protocols" / "deterministic_balance"
config_path = benchmark_dir / "config" / "base.json"
definition_path = deterministic_dir / "definition.json"
definition = load_deterministic_definition(config_path, definition_path)
frozen_inflows = load_frozen_inflows(
    benchmark_dir / "inputs" / "inflows.csv", definition
)
deterministic_run = run_deterministic_benchmark(definition, frozen_inflows)
print(f"Cenários: {len(definition.scenarios)}")
print(f"Casos determinísticos: {len(deterministic_run.cases)}")

## Resultados por cenário

Para cada cenário, o primeiro painel mostra o percentual de energia entregue na matriz 6×6. O segundo compara as trajetórias de potência e nível de montante nos casos `off/off`, `minimum/minimum`, `half/half` e `maximum/maximum`.

In [ ]:
summary = pd.read_csv(deterministic_dir / "reference_results" / "balance_summary.csv")
figures_dir = deterministic_dir / "figures"
for scenario in summary["scenario"].drop_duplicates():
    print(f"Cenário: {scenario}")
    display(Image(filename=figures_dir / f"{scenario}_factorial_heatmap.png"))
    display(Image(filename=figures_dir / f"{scenario}_representative_trajectories.png"))

## Comparação opcional com o GIVP

A referência acima não depende do otimizador. A célula seguinte só executa o GIVP quando `RUN_GIVP_COMPARISON` é alterado para `True`; seus resultados são locais e nunca sobrescrevem os CSVs canônicos.

In [ ]:
RUN_GIVP_COMPARISON = False
if RUN_GIVP_COMPARISON:
    from givp.examples.synthetic_hydropower.runner import (
        ExperimentConfig,
        optimize_scenario,
    )

    typical = next(
        item for item in definition.scenarios if item.definition.name == "typical"
    )
    experiment = ExperimentConfig(
        cascade=definition.cascade,
        scenarios={"typical": typical.definition},
        optimizer={
            "max_iterations": 40,
            "vnd_iterations": 15,
            "ils_iterations": 5,
        },
    )
    optimized = optimize_scenario(experiment, "typical", typical.seed)
    matrix_best = summary.loc[
        summary["scenario"] == "typical", "delivered_energy_mwh"
    ].max()
    print({"givp_energy_mwh": optimized.energy_mwh, "matrix_best_mwh": matrix_best})